In [24]:
import pandas as pd


df = pd.read_csv('C:/Users/swarn/OneDrive/Documents/wipro 24 Agriculture Yield Prediction System/Dataset/Raw Data/yield_df.csv')
print(df.head())

      Area         Item  Year  hg_ha_yield  average_rain_fall_mm_per_year  \
0  Albania        Maize  1990        36613                           1485   
1  Albania     Potatoes  1990        66667                           1485   
2  Albania  Rice, paddy  1990        23333                           1485   
3  Albania      Sorghum  1990        12500                           1485   
4  Albania     Soybeans  1990         7000                           1485   

   pesticides_tonnes  avg_temp  
0              121.0     16.37  
1              121.0     16.37  
2              121.0     16.37  
3              121.0     16.37  
4              121.0     16.37  


In [25]:
import snowflake.connector
import pandas as pd

conn = snowflake.connector.connect(
    user="swarnatripathy",
    password="ssu6GLQVaBgcFmF",
    account="zbtaykt-kt72506",
    warehouse="COMPUTE_WH",
    database="AGRICULTURE_DB",
    schema="RAW_DATA"
)

# Create cursor
cur = conn.cursor()
cur.execute("SELECT * FROM YIELD_DF LIMIT 10")
df = pd.DataFrame(
    cur.fetchall(),
    columns=[desc[0] for desc in cur.description]
)
df

,INDEX_COL,AREA,ITEM,YEAR,HG_HA_YIELD,AVERAGE_RAIN_FALL_MM_PER_YEAR,PESTICIDES_TONNES,AVG_TEMP
0,None,Albania,Maize,1990,36613.0,1485.0,121.0,16.37
1,None,Albania,Potatoes,1990,66667.0,1485.0,121.0,16.37
2,None,Albania,"Rice, paddy",1990,23333.0,1485.0,121.0,16.37
3,None,Albania,Sorghum,1990,12500.0,1485.0,121.0,16.37
4,None,Albania,Soybeans,1990,7000.0,1485.0,121.0,16.37
5,None,Albania,Wheat,1990,30197.0,1485.0,121.0,16.37
6,None,Albania,Maize,1991,29068.0,1485.0,121.0,15.36
7,None,Albania,Potatoes,1991,77818.0,1485.0,121.0,15.36
8,None,Albania,"Rice, paddy",1991,28538.0,1485.0,121.0,15.36
9,None,Albania,Sorghum,1991,6667.0,1485.0,121.0,15.36


In [26]:
import pandas as pd
import numpy as np

def transform_data(df):
    
    # Standardize column names 
    df.columns = df.columns.str.strip().str.upper()
    
    # Convert numeric columns properly
    numeric_cols = [
        "YEAR",
        "HG_HA_YIELD",
        "AVERAGE_RAIN_FALL_MM_PER_YEAR",
        "PESTICIDES_TONNES",
        "AVG_TEMP"
    ]
    
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    
    # Handle missing values
    df = df.fillna(0)
    
    # Feature Engineering
    
    # Rainfall-Yield ratio
    df["RAIN_YIELD_RATIO"] = (
        df["HG_HA_YIELD"] / (df["AVERAGE_RAIN_FALL_MM_PER_YEAR"] + 1)
    )
    
    # Temperature-Yield ratio
    df["TEMP_YIELD_RATIO"] = (
        df["HG_HA_YIELD"] / (df["AVG_TEMP"] + 1)
    )
    
    # High yield flag
    df["HIGH_YIELD_FLAG"] = np.where(
        df["HG_HA_YIELD"] > df["HG_HA_YIELD"].mean(),
        1,
        0
    )
    
    print("Transformation completed successfully.")
    
    return df


# Apply transformation
df = transform_data(df)

df.head()

Transformation completed successfully.


,INDEX_COL,AREA,ITEM,YEAR,HG_HA_YIELD,AVERAGE_RAIN_FALL_MM_PER_YEAR,PESTICIDES_TONNES,AVG_TEMP,RAIN_YIELD_RATIO,TEMP_YIELD_RATIO,HIGH_YIELD_FLAG
0,0,Albania,Maize,1990,36613.0,1485.0,121.0,16.37,24.638627,2107.829591,1
1,0,Albania,Potatoes,1990,66667.0,1485.0,121.0,16.37,44.863392,3838.054116,1
2,0,Albania,"Rice, paddy",1990,23333.0,1485.0,121.0,16.37,15.701884,1343.293034,0
3,0,Albania,Sorghum,1990,12500.0,1485.0,121.0,16.37,8.411844,719.631549,0
4,0,Albania,Soybeans,1990,7000.0,1485.0,121.0,16.37,4.710633,402.993667,0


In [27]:
from snowflake.connector.pandas_tools import write_pandas

success, nchunks, nrows, _ = write_pandas(
    conn,
    df,
    "YIELD_DF_TRANSFORMED",
    auto_create_table=True
)

print("Success:", success)
print("Rows inserted:", nrows)

Success: True
Rows inserted: 10


In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   INDEX_COL                      10 non-null     object 
 1   AREA                           10 non-null     str    
 2   ITEM                           10 non-null     str    
 3   YEAR                           10 non-null     int64  
 4   HG_HA_YIELD                    10 non-null     float64
 5   AVERAGE_RAIN_FALL_MM_PER_YEAR  10 non-null     float64
 6   PESTICIDES_TONNES              10 non-null     float64
 7   AVG_TEMP                       10 non-null     float64
 8   RAIN_YIELD_RATIO               10 non-null     float64
 9   TEMP_YIELD_RATIO               10 non-null     float64
 10  HIGH_YIELD_FLAG                10 non-null     int64  
dtypes: float64(6), int64(2), object(1), str(2)
memory usage: 1.1+ KB


In [29]:
# Standardize column names 
df.columns = df.columns.str.upper()

# Convert numeric columns 
df["YEAR"] = pd.to_numeric(df["YEAR"], errors="coerce")
df["HG_HA_YIELD"] = pd.to_numeric(df["HG_HA_YIELD"], errors="coerce")
df["AVERAGE_RAIN_FALL_MM_PER_YEAR"] = pd.to_numeric(df["AVERAGE_RAIN_FALL_MM_PER_YEAR"], errors="coerce")
df["PESTICIDES_TONNES"] = pd.to_numeric(df["PESTICIDES_TONNES"], errors="coerce")
df["AVG_TEMP"] = pd.to_numeric(df["AVG_TEMP"], errors="coerce")

# Fill nulls
df = df.fillna(0)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   INDEX_COL                      10 non-null     object 
 1   AREA                           10 non-null     str    
 2   ITEM                           10 non-null     str    
 3   YEAR                           10 non-null     int64  
 4   HG_HA_YIELD                    10 non-null     float64
 5   AVERAGE_RAIN_FALL_MM_PER_YEAR  10 non-null     float64
 6   PESTICIDES_TONNES              10 non-null     float64
 7   AVG_TEMP                       10 non-null     float64
 8   RAIN_YIELD_RATIO               10 non-null     float64
 9   TEMP_YIELD_RATIO               10 non-null     float64
 10  HIGH_YIELD_FLAG                10 non-null     int64  
dtypes: float64(6), int64(2), object(1), str(2)
memory usage: 1.1+ KB


In [31]:
import pandas as pd

# Load raw CSV
df = pd.read_csv(r'C:/Users/swarn/OneDrive/Documents/wipro 24 Agriculture Yield Prediction System/Dataset/Raw Data/yield_df.csv')

# Remove unwanted index column (like Unnamed: 0)
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Remove extra spaces from column names
df.columns = df.columns.str.strip()

print("Original Columns:")
print(df.columns)

# Force rename ALL columns by position
df.columns = [
    "Area",
    "Item",
    "Year",
    "Yield",
    "Rainfall",
    "Pesticides",
    "Temperature"
]

print("New Columns:")
print(df.columns)

# Save cleaned CSV
df.to_csv(
    r'C:/Users/swarn/OneDrive/Documents/wipro 24 Agriculture Yield Prediction System/Dataset/Processed Data/cleaned_data.csv',
    index=False
)

print("FINAL CLEAN CSV CREATED")

Original Columns:
Index(['Area', 'Item', 'Year', 'hg_ha_yield', 'average_rain_fall_mm_per_year',
       'pesticides_tonnes', 'avg_temp'],
      dtype='str')
New Columns:
Index(['Area', 'Item', 'Year', 'Yield', 'Rainfall', 'Pesticides',
       'Temperature'],
      dtype='str')
FINAL CLEAN CSV CREATED
